# Proposed Framework Walkthrough

This notebook is a compact walkthrough of the proposed online bin-packing framework. It connects the main project components in the same order they are used during validation:

- instantiate the Gymnasium packing environment
- build the policy network and load checkpoint weights when available
- roll out the packing policy until the bin reaches a blocked state or target utilization
- visualize the packing sequence with an interactive replay
- run MCTS-based safe rearrangement from the blocked state
- optionally optimize and replay the rearrangement sequence

![Proposed method](../figures/Proposed_method.png)


## 1. Imports And Config

The demo uses `configs/test_default.yaml` for reproducible defaults. The default agent is the checkpoint-backed policy, so `CHECKPOINT_PATH` must point to an available policy checkpoint.


In [ ]:
import base64
from copy import deepcopy
from dataclasses import replace
import importlib
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import gymnasium as gym
from IPython.display import IFrame, Markdown, display
from gymnasium.envs.registration import register

import packing.a_star as a_star_module
import packing.mcts as mcts_module
import packing.plans as plans_module
import packing.test_utils as test_utils_module
import packing.visualizer as visualizer_module

importlib.reload(plans_module)
importlib.reload(mcts_module)
importlib.reload(a_star_module)
importlib.reload(visualizer_module)
importlib.reload(test_utils_module)

from packing.agents import PackingAgent
from packing.interactive_replay import InteractiveReplayRecorder
Visualizer = visualizer_module.Visualizer
execution_plan_from_mcts_plan = plans_module.execution_plan_from_mcts_plan
optimize_execution_plan = a_star_module.optimize_execution_plan
mcts = mcts_module.mcts
from packing.debug_utils import print_execution_steps, print_mcts_stats, print_mcts_steps
from packing.policy_loader import build_net, load_policy_weights, resolve_runtime_device, set_eval_seed
load_test_config = test_utils_module.load_test_config
record_pack_step = test_utils_module.record_pack_step
from packing_env.visualization import PackVisualizer


In [ ]:
CONFIG_PATH = PROJECT_ROOT / "configs/test_default.yaml"
CHECKPOINT_PATH = PROJECT_ROOT / "outputs/train_outputs/random/policy_step.pth"
REPLAY_DIR = PROJECT_ROOT / "outputs/plotly_live/notebook_demo"
SEED = 0

config = load_test_config(str(CONFIG_PATH))
config = replace(config, agent="policy")


def display_replay(replay_file: Path, height: int = 780) -> None:
    html_bytes = replay_file.read_bytes()
    data_url = "data:text/html;base64," + base64.b64encode(html_bytes).decode("ascii")
    display(IFrame(src=data_url, width="100%", height=height))


config


## 2. Instantiate The Gym Environment

The environment can be created directly through Gymnasium. It contains the container, buffer, EMS generator, stability validator, and height map.


In [ ]:
if "OnlinePack-v2" not in gym.envs.registry:
    register(id="OnlinePack-v2", entry_point="packing_env:PackingEnv")

set_eval_seed(SEED)
env = gym.make(
    "OnlinePack-v2",
    k_placement=80,
    ds_name=config.ds_name,
    buffer_capacity=12,
    container_size=config.container_size,
    item_buffer_space=config.buffer_space,
    remove_inscribed_ems=config.remove_inscribed_ems,
)
# obs, info = env.reset(seed=SEED)

# print(type(env.unwrapped).__name__)
# print("initial buffer dims:", env.unwrapped.buffer.dims())
# print("initial utilization:", env.unwrapped.container.utilization)


## 3. Build Network And Load Weights

`PackingAgent` is the default policy interface. It wraps the actor and critic networks and exposes a small `predict(obs)` method. The cells below also show the lower-level network construction and checkpoint-loading call explicitly.


In [ ]:
device = resolve_runtime_device(config.device)
actor, critic = build_net(device=str(device))

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Policy checkpoint not found: {CHECKPOINT_PATH}")
load_policy_weights(actor, critic, str(CHECKPOINT_PATH), device)
agent = PackingAgent(
    device=str(device),
    k_placement=env.unwrapped.k_placement,
    checkpoint_path=str(CHECKPOINT_PATH),
)
policy_name = f"policy checkpoint: {CHECKPOINT_PATH}"

print("device:", device)
print("actor:", actor.__class__.__name__)
print("critic:", critic.__class__.__name__)
print("packing policy:", policy_name)


## 4. Policy Packing Rollout

This section mirrors `test.py`'s `pack_until_blocked` workflow, but it is split into smaller notebook cells:

1. reset the environment and initialize replay/log containers
2. define one rollout helper that performs the repeated policy steps
3. run the helper and inspect the resulting pack log

The loop stops when the incoming item is not placeable, when target utilization is reached, or when `config.max_steps` is reached as a safety cap.


In [ ]:
set_eval_seed(SEED)
env.reset(seed=SEED)
packing_env = env.unwrapped
initial_buffer = packing_env.buffer.dims()

print("initial buffer length:", len(initial_buffer))
print("initial item:", initial_buffer[0])
print("target utilization:", config.target_util)
print("max policy steps:", config.max_steps)


### 4.1 Rollout Helper

The helper below keeps the policy loop close to `test.py`: sample the next buffer item, build pack data, call `agent.predict(obs)`, pack the selected box, update the buffer and other related stuff.


In [ ]:
def run_policy_until_blocked(packing_env, agent, config, seed):
    pack_history = []
    pack_log = []
    replay_frames = [("Initial State", deepcopy(packing_env))]

    def result(blocked_item=None, blocked_env=None, target_reached=False):
        return {
            "pack_history": pack_history,
            "pack_log": pack_log,
            "replay_frames": replay_frames,
            "blocked_item": blocked_item,
            "blocked_env": blocked_env,
            "target_reached": target_reached,
        }

    for step in range(config.max_steps):
        item = packing_env.buffer.sample_item()
        obs = packing_env.get_pack_data(item)
        if not obs["placable"]:
            blocked_env = deepcopy(packing_env)
            title = f"Blocked Before MCTS at Pack Step {step} - seed {seed}"
            replay_frames.append((title, deepcopy(packing_env)))
            print(
                "blocked at step={}, utilization={:.4f}, placed={}, unpackable={}".format(
                    step,
                    packing_env.container.utilization,
                    len(packing_env.container.placed_items),
                    len(packing_env.container.unpackable_boxes),
                )
            )
            return result(blocked_item=item, blocked_env=blocked_env)

        box, (_, action_idx, predicted_value) = agent.predict(obs)
        selected_ems = packing_env.ems_list[action_idx % packing_env.k_placement]
        packing_env.pack(box, selected_ems=selected_ems)
        packing_env.buffer.update(item)
        packing_env.validate_packing_state()

        pack_history.append(record_pack_step(box, packing_env))
        title = f"Pack Step {step + 1} - seed {seed}"
        replay_frames.append((title, deepcopy(packing_env)))
        pack_log.append(
            {
                "step": step + 1,
                "action_idx": action_idx,
                "item_dim": item.raw().astype(int).tolist(),
                "box": repr(box),
                "predicted_value": float(predicted_value),
                "utilization": packing_env.container.utilization,
            }
        )
        print(
            "{} step={}, utilization={:.4f}, placed={}".format(
                config.agent,
                step,
                packing_env.container.utilization,
                len(packing_env.container.placed_items),
            )
        )

        if packing_env.container.utilization >= config.target_util:
            print("target utilization reached before MCTS was needed")
            return result(target_reached=True)

    raise AssertionError(f"no blocked item found within {config.max_steps} steps")


### 4.2 Run The Policy

After this cell runs, later sections reuse `pack_history`, `replay_frames`, `blocked_item`, and `blocked_env` for replay and MCTS.


In [ ]:
rollout = run_policy_until_blocked(packing_env, agent, config, SEED)

pack_history = rollout["pack_history"]
pack_log = rollout["pack_log"]
replay_frames = rollout["replay_frames"]
blocked_item = rollout["blocked_item"]
blocked_env = rollout["blocked_env"]
target_reached = rollout["target_reached"]

print("placed items:", len(packing_env.container.placed_items))
print("replay frames:", len(replay_frames))
print("target reached:", target_reached)
print("blocked item:", None if blocked_item is None else blocked_item.raw().astype(int).tolist())


In [ ]:
lines = [
    "| step | item dim | action idx | predicted value | utilization |",
    "|---:|---|---:|---:|---:|",
]
for row in pack_log:
    lines.append(
        f"| {row['step']} | `{row['item_dim']}` | {row['action_idx']} | "
        f"{row['predicted_value']:.4f} | {row['utilization']:.4f} |"
    )
display(Markdown("\n".join(lines)))


## 5. Interactive Policy Replay

The replay below records every packing step as a Plotly frame. Use **Prev**, **Next**, **Play**, or the slider to inspect each packed item. You can still rotate each 3D plot while moving through the frames.


In [ ]:
replay_path = REPLAY_DIR / "packing_steps.html"
recorder = InteractiveReplayRecorder(str(replay_path), interval_ms=700)

replay_visualizer = PackVisualizer(
    replay_frames[0][1],
    title="Notebook Packing Replay",
    show_anchor=True,
    show_ems=False,
)

for frame_idx, (title, snapshot) in enumerate(replay_frames):
    replay_visualizer.env = snapshot
    replay_visualizer.title = title
    if frame_idx == 0:
        replay_visualizer.reset_history()
    fig, _ = replay_visualizer.refresh()
    recorder.capture(title, fig)

recorder.save()
display_replay(replay_path)


## 6. Run MCTS From The Blocked State

This mirrors the MCTS part of `test.py`: use the true blocked item and blocked environment produced by the packing loop, run `mcts(...)`, convert the result to an execution plan, and record replay snapshots if a sequence is found.


In [ ]:
mcts_result = None
mcts_stats = None
execution_plan = None
mcts_replay_path = REPLAY_DIR / "mcts_replay.html"

if blocked_item is None:
    print("MCTS skipped because no blocked item was found.")
else:
    mcts_result, mcts_stats = mcts(
        deepcopy(blocked_env),
        agent,
        new_item=blocked_item,
        iterations=config.iterations,
        max_unpack=config.max_unpack,
        Uti_requirement=config.target_util,
        return_info=True,
        max_child=config.mcts_max_child,
    )
    print_mcts_stats(mcts_stats)

    if mcts_result is None:
        print("MCTS did not find an unpack/repack sequence for this blocked state.")
    else:
        print_mcts_steps(mcts_result)
        execution_plan = execution_plan_from_mcts_plan(mcts_result)
        replay_config = replace(
            config,
            save_replay=True,
            replay_path=str(mcts_replay_path),
        )
        replay_helper = Visualizer(replay_config)
        replay_helper.save_sequence_replay(
            seed=SEED,
            initial_buffer=initial_buffer,
            pack_history=pack_history,
            mcts_used=True,
            execution_plan=execution_plan,
            step_prefix="MCTS Step",
            start_from_blocked=True,
        )
        print("MCTS replay saved to:", mcts_replay_path)


## 7. Interactive MCTS Replay

If MCTS found a plan, this replay starts at the blocked state and then shows each unpack and pack operation. Use the controls to inspect how the rearrangement creates space for the incoming item.


In [ ]:
if execution_plan is None:
    print("No MCTS replay to show.")
elif not mcts_replay_path.exists():
    print(f"Replay file was not found: {mcts_replay_path}")
else:
    display_replay(mcts_replay_path)


## 8. Optimize Operation Sequence

The raw MCTS plan lists unpack operations first and pack operations second. The optimizer tries to shorten that into an executable sequence, for example by replacing an unpack-then-pack pair with one direct repack operation. The optimized sequence is saved as a separate interactive replay.


In [ ]:
optimized_execution_plan = None
optimized_replay_path = REPLAY_DIR / "mcts_optimized_replay.html"

if mcts_result is None or blocked_env is None:
    print("No MCTS result to optimize.")
else:
    optimized_execution_plan = optimize_execution_plan(blocked_env, mcts_result)
    print_execution_steps(optimized_execution_plan)

    optimized_replay_config = replace(
        config,
        save_replay=True,
        replay_path=str(optimized_replay_path),
    )
    optimized_replay_helper = Visualizer(optimized_replay_config)
    optimized_replay_helper.save_sequence_replay(
        seed=SEED,
        initial_buffer=initial_buffer,
        pack_history=pack_history,
        mcts_used=True,
        execution_plan=optimized_execution_plan,
        step_prefix="Optimized Step",
        start_from_blocked=True,
    )
    print("Optimized replay saved to:", optimized_replay_path)


## 9. Interactive Optimized Replay

This replay starts at the blocked state and shows the optimized execution order. It should usually have the same final packed state as the raw MCTS replay, but fewer or clearer operations.


In [ ]:
if optimized_execution_plan is None:
    print("No optimized replay to show.")
elif not optimized_replay_path.exists():
    print(f"Replay file was not found: {optimized_replay_path}")
else:
    display_replay(optimized_replay_path)


## 10. Command-Line Smoke Demo

For the automated smoke workflow, use the script-level demo config:

```bash
python test.py --config configs/test_default.yaml --replay-path outputs/plotly_live/demo
```

The notebook above is more explicit because it shows the environment, network, policy, packing loop, MCTS search, and visualization objects separately.
